# Structural Credit Portfolio Model

This notebook develops an international credit portfolio model combining:

- Merton structural credit modelling
- real-world issuer default probabilities
- multifactor Student-t dependence
- default and rating-migration risk
- portfolio VaR, Expected Shortfall and risk contributions

The model is first demonstrated for Apple and subsequently extended to a
3-country × 3-industry portfolio.v

## 1. Single-Issuer Merton Application: Apple

We begin by estimating Apple's equity volatility from historical market data.
The valuation date is fixed at 30 June 2026 to ensure reproducibility.

In [1]:
import numpy as np
import yfinance as yf


ticker = "AAPL"
valuation_date = "2025-12-31"

# Include the final trading day of 2024 to calculate the first return of 2025.
# The yfinance end date is exclusive.
market_data = yf.download(
    ticker,
    start="2024-12-31",
    end="2026-01-01",
    auto_adjust=False,
    progress=False,
)

# Adjusted prices are appropriate for estimating equity returns and volatility.
adjusted_prices = market_data["Adj Close"].squeeze().dropna()

# Calculate returns first and then retain returns occurring during 2025.
log_returns = np.log(adjusted_prices / adjusted_prices.shift(1))
log_returns_2025 = log_returns.loc["2025-01-01":"2025-12-31"].dropna()

# Annualized equity volatility based on daily log returns.
sigma_E = log_returns_2025.std(ddof=1) * np.sqrt(252)

# Use the unadjusted closing price for the later equity-value calculation.
closing_prices = market_data["Close"].squeeze().dropna()
year_end_price = closing_prices.loc[:valuation_date].iloc[-1]
actual_price_date = closing_prices.loc[:valuation_date].index[-1]

print(f"Ticker: {ticker}")
print(f"Return observations in 2025: {len(log_returns_2025)}")
print(f"Final price date: {actual_price_date.date()}")
print(f"Year-end share price: ${year_end_price:.2f}")
print(f"Annualized equity volatility: {sigma_E:.2%}")

Ticker: AAPL
Return observations in 2025: 250
Final price date: 2025-12-31
Year-end share price: $271.86
Annualized equity volatility: 32.21%


In [2]:
# Retrieve Apple's historical shares outstanding.
apple = yf.Ticker(ticker)

shares_history = apple.get_shares_full(
    start="2025-01-01",
    end="2026-01-02",
).dropna().sort_index()

# Select the latest share count available on or before the valuation date.
shares_at_valuation = shares_history.loc[:valuation_date]

shares_outstanding = shares_at_valuation.iloc[-1]
shares_date = shares_at_valuation.index[-1]

# Equity market value used in the Merton model.
equity_value = year_end_price * shares_outstanding

print(f"Shares date: {shares_date.date()}")
print(f"Shares outstanding: {shares_outstanding / 1e9:.3f} billion")
print(f"Equity market value: ${equity_value / 1e9:.2f} billion")

Shares date: 2025-11-17
Shares outstanding: 14.840 billion
Equity market value: $4034.51 billion


### Default point

The Merton model requires a debt threshold below which the company is assumed to
default at the one-year horizon. Because this threshold is not directly observable,
we use the common KMV-style approximation:

\[
D = \text{short-term debt} + 0.5 \times \text{long-term debt}.
\]

This is a modelling proxy rather than the company’s total debt or a contractual
default boundary.

In [3]:
import pandas as pd
import yfinance as yf

apple = yf.Ticker(ticker)

# Retrieve Apple's quarterly balance sheet.
balance_sheet = apple.quarterly_balance_sheet.copy()
balance_sheet.columns = pd.to_datetime(balance_sheet.columns)

# Retrieve Apple's quarterly balance sheet.
balance_sheet = apple.quarterly_balance_sheet.copy()
balance_sheet.columns = pd.to_datetime(balance_sheet.columns)

# Select the latest financial statement available before the valuation date.
available_dates = balance_sheet.columns[
    balance_sheet.columns <= pd.Timestamp(valuation_date)
]
statement_date = available_dates.max()

short_term_debt = balance_sheet.loc["Current Debt", statement_date]
long_term_debt = balance_sheet.loc["Long Term Debt", statement_date]

# KMV-style default point.
debt_default_point = short_term_debt + 0.5 * long_term_debt

print(f"Statement date: {statement_date.date()}")
print(f"Short-term debt: ${short_term_debt / 1e9:.2f} billion")
print(f"Long-term debt: ${long_term_debt / 1e9:.2f} billion")
print(f"Default point D: ${debt_default_point / 1e9:.2f} billion")

Statement date: 2025-12-31
Short-term debt: $13.82 billion
Long-term debt: $76.69 billion
Default point D: $52.17 billion


In [4]:
# Retrieve the one-year US Treasury yield from FRED.
fred_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=DGS1&cosd=2025-12-01&coed=2025-12-31"
)

risk_free_data = pd.read_csv(fred_url)

# Standardize the date-column name and data types.
risk_free_data = risk_free_data.rename(
    columns={risk_free_data.columns[0]: "Date"}
)
risk_free_data["Date"] = pd.to_datetime(risk_free_data["Date"])
risk_free_data["DGS1"] = pd.to_numeric(
    risk_free_data["DGS1"],
    errors="coerce",
)

# Select the latest available observation on or before the valuation date.
available_rates = risk_free_data.loc[
    risk_free_data["Date"] <= pd.Timestamp(valuation_date)
].dropna(subset=["DGS1"])

risk_free_observation = available_rates.iloc[-1]

rate_date = risk_free_observation["Date"]
treasury_yield_percent = risk_free_observation["DGS1"]

# Convert the percentage yield into the decimal rate required by the model.
risk_free_rate = treasury_yield_percent / 100

print(f"Risk-free-rate date: {rate_date.date()}")
print(f"One-year Treasury yield: {treasury_yield_percent:.2f}%")
print(f"Risk-free model input: {risk_free_rate:.4f}")

Risk-free-rate date: 2025-12-31
One-year Treasury yield: 3.48%
Risk-free model input: 0.0348


In [5]:
from quant_finance.credit.merton_calibration import calibrate_merton


time_to_maturity = 1.0

asset_value, asset_volatility = calibrate_merton(
    equity_value=equity_value,
    equity_volatility=sigma_E,
    debt=debt_default_point,
    risk_free_rate=risk_free_rate,
    maturity=time_to_maturity,
)

print(f"Calibrated asset value: ${asset_value / 1e9:.2f} billion")
print(f"Calibrated asset volatility: {asset_volatility:.2%}")

Calibrated asset value: $4084.89 billion
Calibrated asset volatility: 31.82%


In [6]:
from quant_finance.credit.merton import (
    distance_to_default,
    merton_probability_of_default,
)


distance_to_default_q = distance_to_default(
    asset_value=asset_value,
    debt_face_value=debt_default_point,
    risk_free_rate=risk_free_rate,
    asset_volatility=asset_volatility,
    time_to_maturity=time_to_maturity,
)

probability_of_default_q = merton_probability_of_default(
    asset_value=asset_value,
    debt_face_value=debt_default_point,
    risk_free_rate=risk_free_rate,
    asset_volatility=asset_volatility,
    time_to_maturity=time_to_maturity,
)

print(f"Risk-neutral distance to default: {distance_to_default_q:.4f}")
print(f"Risk-neutral probability of default: {probability_of_default_q:.3e}")

Risk-neutral distance to default: 13.6560
Risk-neutral probability of default: 9.294e-43


In [7]:
from quant_finance.credit.merton_calibration import (
    merton_calibration_residuals,
)


calibration_residuals = merton_calibration_residuals(
    unknowns=np.array([asset_value, asset_volatility]),
    equity_value=equity_value,
    equity_volatility=sigma_E,
    debt=debt_default_point,
    risk_free_rate=risk_free_rate,
    maturity=time_to_maturity,
)

equity_value_error = calibration_residuals[0]
equity_volatility_error = calibration_residuals[1]

relative_equity_value_error = equity_value_error / equity_value
relative_equity_volatility_error = equity_volatility_error / sigma_E

print(f"Equity-value residual: ${equity_value_error:,.6f}")
print(f"Relative equity-value error: {relative_equity_value_error:.3e}")
print(f"Equity-volatility residual: {equity_volatility_error:.3e}")
print(
    "Relative equity-volatility error: "
    f"{relative_equity_volatility_error:.3e}"
)

Equity-value residual: $0.000000
Relative equity-value error: 0.000e+00
Equity-volatility residual: -2.680e-11
Relative equity-volatility error: -8.321e-11
